In [5]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

current_dir = Path.cwd()

if current_dir.name == "evaluate":
    PROJECT_ROOT = current_dir.parents[1]
else:
    PROJECT_ROOT = current_dir

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv()


True

In [ ]:
from sentence_transformers import SentenceTransformer
from src.config import EMBEDDING_MODEL, GOLD_KAGGLE_BM25_MODEL
from src.recommendation.core.model_bm25 import BM25PlusRecommender
from src.recommendation.core.recommend import load_default_runtime_indexes
from src.storage.minio_client import get_minio_client, download_pickle


client = get_minio_client()
bm25_model = download_pickle(client, GOLD_KAGGLE_BM25_MODEL)

emb_model = SentenceTransformer(EMBEDDING_MODEL)

runtime_indexes = load_default_runtime_indexes()


Đã load Pickle: s3://bigdata-nhom6/gold/kaggle/bm25/bm25_model.pkl (251.6 MB)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1570.49it/s]


[OK] Loaded runtime index: kaggle
     Metadata rows : 1280299
     Title vectors : 1280299
     Skills vectors: 1280299


In [7]:
import importlib
import pandas as pd
from src.recommendation.core.hybrid_rrf import reciprocal_rank_fusion

hybrid = importlib.import_module("scripts.recommend.03_hybrid_rrf")
get_emb_missing_skills = hybrid.get_emb_missing_skills

role = "Data Engineer"
skills = ["Python", "SQL", "Git"]
print(f"Test query: {role} | {skills}\n")

res_bm25 = bm25_model.query(role, skills, 10)
res_emb = get_emb_missing_skills(emb_model, runtime_indexes, role, skills)
res_hybrid = reciprocal_rank_fusion(res_bm25, res_emb, 10)

df = pd.DataFrame(res_hybrid)
if not df.empty:
    display(df[['skill', 'rrf_score', 'job_count', 'bm25_rank', 'emb_rank']])
else:
    print("No result")

Test query: Data Engineer | ['Python', 'SQL', 'Git']



KeyboardInterrupt: 

In [ ]:
import random
import importlib
import pandas as pd
from tqdm import tqdm

hybrid = importlib.import_module("scripts.recommend.03_hybrid_rrf")
get_emb = hybrid.get_emb_missing_skills
from src.recommendation.core.hybrid_rrf import reciprocal_rank_fusion

def split_data(skills, ratio=0.6):
    random.shuffle(skills)
    idx = max(1, int(len(skills) * ratio))
    return skills[:idx], set(skills[idx:])

def get_metrics(preds, targets):
    if not targets or not preds: return 0, 0, 0
    hits = sum(1 for p in preds if p in targets)
    
    r = hits / len(targets)
    p = hits / len(preds)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return r, p, f1

# load & prep data
df = pd.read_parquet(PROJECT_ROOT / "data/runtime_index/kaggle/jobs_metadata.parquet")
df['clean_skills'] = df['skills_normalized'].apply(lambda x: [s.strip() for s in str(x).split(',') if s.strip()])

# lọc job có >= 4 skill rồi bốc 200 samples
test_df = df[df['clean_skills'].apply(len) >= 4].sample(n=500, random_state=42)

# dict lưu điểm [recall, precision, f1]
scores = {"bm25": [0,0,0], "emb": [0,0,0], "hybrid": [0,0,0]}
valid = 0

for _, row in tqdm(test_df.iterrows(), total=500):
    given, target = split_data(row['clean_skills'])
    if not target: continue
    valid += 1
    
    title = row['title_core']
    
    # query 3 models
    bm25_raw = bm25_model.query(title, given, 10)
    bm25_preds = [x['skill'] for x in bm25_raw]
    
    emb_df = get_emb(emb_model, runtime_indexes, title, given)
    emb_preds = emb_df['skill'].tolist() if not emb_df.empty else []
    
    hybrid_preds = [x['skill'] for x in reciprocal_rank_fusion(bm25_raw, emb_df, 20)]

    # tính điểm cộng dồn
    for name, preds in [("bm25", bm25_preds), ("emb", emb_preds), ("hybrid", hybrid_preds)]:
        r, p, f1 = get_metrics(preds, target)
        scores[name][0] += r
        scores[name][1] += p
        scores[name][2] += f1

print(f"\n--- Kết quả evaluate trên {valid} jobs ---")
for m in ["bm25", "emb", "hybrid"]:
    r = (scores[m][0] / valid) * 100
    p = (scores[m][1] / valid) * 100
    f1 = (scores[m][2] / valid) * 100
    print(f"{m.upper():<8} | Recall: {r:>5.2f}% | Precision: {p:>5.2f}% | F1: {f1:>5.2f}%")

 50%|█████     | 100/200 [04:20<04:20,  2.60s/it]


--- Kết quả evaluate trên 100 jobs ---
BM25     | Recall: 34.63% | Precision: 16.30% | F1: 21.20%
EMB      | Recall: 36.74% | Precision: 17.20% | F1: 22.36%
HYBRID   | Recall: 38.97% | Precision: 18.30% | F1: 23.75%
